In [1]:
using PhDProject

In [118]:
function ohmic_spectral_function(w,s,ωc,α,β,thermal_chain_number)
    
    n = 1 ./(exp.(β*w).-1) #Bosonic occupation number
    if thermal_chain_number == 1
        renorm = 1 .+ n
    elseif thermal_chain_number == 2
        renorm = n
    end

    support = exp.(-w/ωc)
    ρ = (w.^(s) .*support).*renorm
    J =  α*(ωc^(1-s))*ρ
    replace_nan(J)
    return J
end
function ohmic_chain_mapping(thermal_chain_number,N_bath,ωc,s,α,β,D;kwargs...)

    N_chain = get(kwargs,:N_chain,N_bath)
    @assert(N_chain <= N_bath)

    couplings,energies = complex(zeros(N_bath)),complex(zeros(N_bath))
    spec_fun(t) = ohmic_spectral_function(t,s,ωc,α,β,thermal_chain_number)
    supp = (0,D)
    my_meas = PhDProject.Measure("my_meas", spec_fun, supp, false, Dict())
    my_op = PhDProject.OrthoPoly("my_op", N_chain-1, my_meas; Nquad=100000);
    α_coeffs,β_coeffs = PhDProject.coeffs(my_op)[:,1],PhDProject.coeffs(my_op)[:,2]
        

    if length(α_coeffs)<N_bath
        energies[1:N_chain] = α_coeffs
        couplings[1:N_chain] = sqrt.(β_coeffs)
        energies[N_chain+1:end] .= α_coeffs[N_chain]
        couplings[N_chain+1:end] .= sqrt(β_coeffs[N_chain])
    else
        energies = α_coeffs
        couplings = sqrt.(β_coeffs)
    end
    return energies,couplings
end
function initialise_choi_state(modes)
    #ψ_init = PhDProject.MPS(ComplexF64,modes,"0")

    gates = PhDProject.ITensor[]

    for i=1:N_ring
        sys = 2*i-1
        anc = 2*i
 
        # Hadamard on system qubit
        push!(gates, PhDProject.op("H", modes[sys]))

        # CNOT: control = system, target = ancilla
        push!(gates, PhDProject.op("CNOT", modes[sys], modes[anc]))
    end
    return gates
end
function initialise_setup(N_ring,J,B,compute_maps_bool,N_bath,ωc,s,α,β,D;kwargs...)
        
    if compute_maps_bool ==true
        qS = 1:2:(2*N_ring)
        qA = 2:2:(2*N_ring)
        q = 1:2*N_ring
    else
        qS = 1:N_ring
        q = qS
    end

    #Creating chain coefficients
    energies_1,couplings_1 = ohmic_chain_mapping(1,N_bath,ωc,s,α,β,D;kwargs...)
    energies_2,couplings_2 = ohmic_chain_mapping(2,N_bath,ωc,s,α,β,D;kwargs...)

    ##Creating indices
    sys_modes = PhDProject.siteinds("S=1/2",q[end])
    bath_modes = PhDProject.siteinds("Boson",2*N_bath)
    modes = [sys_modes;bath_modes]

    MPO_terms = PhDProject.OpSum()
    #Create system hamiltonian MPO terms
    for i =1:N_ring
        MPO_terms += B,"Sz",qS[i]
        if i <N_ring
            MPO_terms += J,"Sx",qS[i],"Sx",qS[i+1]
            MPO_terms += J,"Sy",qS[i],"Sy",qS[i+1]
        else
            @assert(i==N_ring)
            MPO_terms += J,"Sx",qS[i],"Sx",qS[1]
            MPO_terms += J,"Sy",qS[i],"Sy",qS[1]
        end
    end

    #Create Chain mapping MPO terms
    for i=1:N_bath
        MPO_terms += energies_1[i],"Adag",q[end]+(2*i-1),"A",q[end]+(2*i-1) ##energies for chain 1
        MPO_terms -= energies_2[i],"Adag",q[end]+(2*i),"A",q[end]+(2*i) ##energies for chain 2

        if i <N_bath
            MPO_terms += couplings_1[i+1],"Adag",q[end]+(2*i-1),"A",q[end]+(2*i+1) ##hopping site i to i+1 (chain 1)
            MPO_terms += conj(couplings_1[i+1]),"Adag",q[end]+(2*i+1),"A",q[end]+(2*i-1) ##hopping site i+1 to i (chain 1)

            MPO_terms -= couplings_2[i+1],"Adag",q[end]+(2*i),"A",q[end]+(2*i+2) ##hopping site i to i+1 (chain 2)
            MPO_terms -= conj(couplings_2[i+1]),"Adag",q[end]+(2*i+2),"A",q[end]+(2*i) ##hopping site i+1 to i (chain 2)
        end
    end

    #Create coupling MPO term
    MPO_terms += couplings_1[1],"Sx",qS[N_ring],"A",q[end]+1
    MPO_terms += couplings_1[1],"Sx",qS[N_ring],"Adag",q[end]+1

    MPO_terms -= couplings_2[1],"Sx",qS[N_ring],"A",q[end]+2
    MPO_terms -= couplings_2[1],"Sx",qS[N_ring],"Adag",q[end]+2
    #Create MPO
    H_MPO = PhDProject.MPO(MPO_terms,modes)

    #Create initial state
    psi = PhDProject.MPS(ComplexF64,modes,"0")
    gates = initialise_choi_state(modes)
    psi = PhDProject.apply(gates,psi)
    return psi,H_MPO
end

#spin ring parameters
N_ring = 2
J = 1
B = 1
compute_maps_bool = true

#Chain mapping parameters
N_bath = 50
N_chain = 50
ωc = 1 # exponential cutoff
s = 1 #ohmicity
α = 0.05 #coupling strength
β = 1 #inverse temperature
D = 1 #bandwidth

#tdvp_parameters
tdvp_cutoff = 1e-10 #Numerical cutoff for tdvp
minbonddim = 5      #Minimum bond dimension for tdvp 
maxbonddim = 20       #Maximum bond dimension for tdvp
δt = 0.1            #Time step
T = 2               #Evolution time
TDVP_nsite = 2

# #Enrichment parameters
# T_enrich = 0                #Time in simulation where enrichment is used
# Kr_cutoff = 1e-10 #Numerical cutoff used for Krylov enrichment
# k1 = 3                     #Number of Krylov states used
# τ_Krylov = 1


ψ_init,H_MPO = initialise_setup(N_ring,J,B,compute_maps_bool,N_bath,ωc,s,α,β,D;N_chain)
@show(PhDProject.norm(psi))

# TDVP_nsite == 1 && growMPS!(ψ, minbonddim)

obs = PhDProject.Observer(
"times" => current_time,
"σz" => measure_pauli_z, 
"σx" => measure_pauli_x, 
"σy" => measure_pauli_y, 
"SvN" => measure_SvN)

ψ = PhDProject.tdvp(H_MPO, -im * T, ψ_init; time_step = -im * δt, cutoff = tdvp_cutoff, 
mindim = minbonddim, maxdim = maxbonddim, outputlevel = 1, 
observer! = obs, nsite = TDVP_nsite)

PhDProject.norm(psi) = 0.9999999999999999
After sweep 1: maxlinkdim=5 maxerr=1.25E-15 current_time=0.0 - 0.1im time=14.905
After sweep 2: maxlinkdim=5 maxerr=1.39E-12 current_time=0.0 - 0.2im time=2.725
After sweep 3: maxlinkdim=5 maxerr=2.65E-11 current_time=0.0 - 0.3im time=2.335
After sweep 4: maxlinkdim=6 maxerr=4.67E-12 current_time=0.0 - 0.4im time=2.104
After sweep 5: maxlinkdim=6 maxerr=3.06E-11 current_time=0.0 - 0.5im time=2.097
After sweep 6: maxlinkdim=7 maxerr=6.83E-11 current_time=0.0 - 0.6im time=2.104
After sweep 7: maxlinkdim=8 maxerr=7.72E-12 current_time=0.0 - 0.7im time=2.102
After sweep 8: maxlinkdim=8 maxerr=2.18E-11 current_time=0.0 - 0.8im time=2.089
After sweep 9: maxlinkdim=8 maxerr=5.51E-11 current_time=0.0 - 0.9im time=2.102
After sweep 10: maxlinkdim=8 maxerr=5.63E-11 current_time=0.0 - 1.0im time=2.097
After sweep 11: maxlinkdim=8 maxerr=5.05E-11 current_time=0.0 - 1.1im time=2.103
After sweep 12: maxlinkdim=8 maxerr=9.09E-11 current_time=0.0 - 1.2im time=

104-element ITensorMPS.MPS:
 ((dim=2|id=200|"Link,l=1"), (dim=2|id=209|"S=1/2,Site,n=1"))
 ((dim=4|id=4|"Link,l=2"), (dim=2|id=869|"S=1/2,Site,n=2"), (dim=2|id=200|"Link,l=1"))
 ((dim=8|id=689|"Link,l=3"), (dim=2|id=147|"S=1/2,Site,n=3"), (dim=4|id=4|"Link,l=2"))
 ((dim=8|id=781|"Link,l=4"), (dim=2|id=746|"S=1/2,Site,n=4"), (dim=8|id=689|"Link,l=3"))
 ((dim=9|id=377|"Link,l=5"), (dim=2|id=276|"Boson,Site,n=1"), (dim=8|id=781|"Link,l=4"))
 ((dim=6|id=959|"Link,l=6"), (dim=2|id=307|"Boson,Site,n=2"), (dim=9|id=377|"Link,l=5"))
 ((dim=5|id=709|"Link,l=7"), (dim=2|id=150|"Boson,Site,n=3"), (dim=6|id=959|"Link,l=6"))
 ((dim=5|id=542|"Link,l=8"), (dim=2|id=78|"Boson,Site,n=4"), (dim=5|id=709|"Link,l=7"))
 ((dim=5|id=1|"Link,l=9"), (dim=2|id=602|"Boson,Site,n=5"), (dim=5|id=542|"Link,l=8"))
 ((dim=5|id=785|"Link,l=10"), (dim=2|id=154|"Boson,Site,n=6"), (dim=5|id=1|"Link,l=9"))
 ((dim=5|id=768|"Link,l=11"), (dim=2|id=58|"Boson,Site,n=7"), (dim=5|id=785|"Link,l=10"))
 ((dim=5|id=185|"Link,l=12"

In [122]:
obs

Row,times,σz,σx,σy,SvN
,Float64,Array…,Array…,Array…,Array…
1,0.1,"[-1.27716e-10, -4.2709e-8]","[7.08102e-21, -1.6614e-18]","[-1.64862e-18, -6.26751e-18]","[1.0, 0.0503874, 1.00001, 0.0022573, 0.00098305, 1.27245e-6, 5.02153e-7, 4.88215e-10, 1.90841e-10, 6.53194e-14 … 7.34507e-15, 7.34507e-15, 7.34507e-15, 7.34507e-15, 7.34507e-15, 7.34507e-15, 7.34507e-15, 7.34507e-15, 6.68928e-15, 6.40685e-15]"
2,0.2,"[-4.27636e-9, -5.43333e-7]","[-4.65013e-11, 1.43881e-9]","[-5.10478e-12, 2.80594e-10]","[1.0, 0.161134, 1.00011, 0.00770484, 0.00340752, 1.26417e-5, 5.03524e-6, 7.93541e-9, 3.12867e-9, 2.41304e-11 … 4.57227e-12, 4.57163e-12, 4.57131e-12, 4.57131e-12, 4.57099e-12, 4.57035e-12, 4.56816e-12, 4.56819e-12, 2.41728e-12, 2.20652e-12]"
3,0.3,"[-4.31447e-8, -2.61488e-6]","[-1.42244e-10, 1.74119e-9]","[-3.11895e-11, 2.91315e-10]","[1.0, 0.308628, 1.0005, 0.0155216, 0.00695017, 5.34791e-5, 2.14582e-5, 6.22134e-8, 2.42506e-8, 3.32062e-10 … 5.37484e-11, 5.37484e-11, 5.37487e-11, 5.37484e-11, 5.37493e-11, 5.37493e-11, 5.3744e-11, 5.37449e-11, 5.2192e-11, 5.10578e-11]"
4,0.4,"[-2.29838e-7, -7.93455e-6]","[1.39105e-10, 1.74981e-9]","[-6.56606e-11, 9.07193e-9]","[1.0, 0.479566, 1.00142, 0.0251977, 0.0114112, 0.000150244, 6.06839e-5, 2.9583e-7, 1.17221e-7, 1.13425e-9 … 1.75383e-10, 1.75383e-10, 1.75383e-10, 1.75383e-10, 1.75382e-10, 1.75377e-10, 1.75342e-10, 1.75347e-10, 1.69417e-10, 1.67557e-10]"
5,0.5,"[-8.44131e-7, -1.84909e-5]","[-1.86639e-9, 6.69897e-8]","[-6.63422e-10, -5.31936e-8]","[1.0, 0.664144, 1.00314, 0.0363259, 0.0166303, 0.000333795, 0.000135667, 1.00383e-6, 3.98887e-7, 3.08422e-9 … 2.55075e-10, 2.55075e-10, 2.55068e-10, 2.55017e-10, 2.55017e-10, 2.54864e-10, 2.54688e-10, 2.54693e-10, 2.3736e-10, 2.3537e-10]"
6,0.6,"[-2.43439e-6, -3.62693e-5]","[-2.99428e-9, 1.66166e-7]","[-1.60013e-9, -9.30702e-8]","[1.0, 0.854458, 1.00589, 0.0485583, 0.0224689, 0.000637235, 0.000260579, 2.73685e-6, 1.09391e-6, 1.0872e-8 … 6.5266e-10, 6.5255e-10, 6.52493e-10, 6.52097e-10, 6.52106e-10, 6.50273e-10, 6.47302e-10, 6.47324e-10, 4.52708e-10, 4.43203e-10]"
7,0.7,"[-5.92245e-6, -6.2899e-5]","[-3.25867e-9, 3.52888e-7]","[-2.41691e-9, -1.05442e-7]","[1.0, 1.04386, 1.00989, 0.0615932, 0.0288049, 0.00109412, 0.000450102, 6.36929e-6, 2.55325e-6, 2.11491e-8 … 6.24505e-10, 6.24414e-10, 6.24364e-10, 6.24072e-10, 6.24074e-10, 6.22952e-10, 6.20654e-10, 6.20681e-10, 4.83938e-10, 4.76624e-10]"
8,0.8,"[-1.26958e-5, -9.93047e-5]","[-3.10465e-9, 4.67581e-7]","[-2.98034e-9, -7.14959e-8]","[1.0, 1.22664, 1.01529, 0.0751708, 0.035531, 0.00173699, 0.000718867, 1.32066e-5, 5.31517e-6, 5.08769e-8 … 6.94447e-10, 6.94342e-10, 6.94276e-10, 6.94003e-10, 6.93998e-10, 6.91823e-10, 6.88918e-10, 6.88944e-10, 5.66736e-10, 5.59239e-10]"
9,0.9,"[-2.46706e-5, -0.000145396]","[-2.6626e-9, 5.95551e-7]","[-3.31431e-9, -5.35551e-9]","[1.0, 1.3979, 1.02218, 0.0890721, 0.0425543, 0.00259626, 0.00108095, 2.50328e-5, 1.01151e-5, 1.17665e-7 … 9.01165e-10, 9.01002e-10, 9.00915e-10, 9.00618e-10, 9.006e-10, 8.95265e-10, 8.91299e-10, 8.91331e-10, 7.67337e-10, 7.592e-10]"
